In [1]:
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from autotsc.models import LokyStackerV7, LokyStackerV7SoftFilterRidge, TSCGlue, LokyStackerV8Base
from autotsc.old_models import StackerV4, LokyStackerV5, LokyStackerV5SoftRF, LokyStackerV6
from autotsc import utils, data_loader
import polars as pl

In [2]:
dataset = "Worms"
dataset = 'Car'
dataset = 'HandOutlines'
# dataset = 'UWaveGestureLibraryY'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)
# print(X_train.dtype)
# print(y_train.dtype)

(1000, 1, 2709) (1000,) (370, 1, 2709) (370,)


In [3]:
#m = TSCGlue(random_state=492, n_repetitions=1, n_jobs=16, keep_features=True, verbose=2)
#m.fit(X_train, y_train)
#y_pred = m.predict(X_test)

In [4]:
m = LokyStackerV8Base(random_state=492, n_repetitions=1, n_jobs=16, keep_features=True, verbose=2)
m.fit(X_train, y_train)

[0.00s] Starting executor with 16 workers, run_dir=./tscglue/8be2aac49ff44aba
[0.02s] Saved X and y to disk in 0.02s
[31.72s] Computed QUANT features (1000, 24015) (183.22 MB) in 31.6978s
[31.80s] Starting repetition 0
[46.66s] Computed MultiRocket features (1000, 49728) (379.39 MB) in 14.8566s
[101.19s] Computed Hydra features (1000, 9216) (70.31 MB) in 54.5273s
[334.65s] Computed RDST features (1000, 30000) (228.88 MB) in 233.4602s
[334.66s] Starting training with 16 workers for 40 models
[344.29s] Trained multirockethydra-bestk-p-ridgecv_r0 in 9.5921s for f-5/r-0 (2.96 MB)
[344.73s] Trained multirockethydra-bestk-p-ridgecv_r0 in 9.9898s for f-8/r-0 (2.96 MB)
[345.02s] Trained multirockethydra-bestk-p-ridgecv_r0 in 10.2489s for f-3/r-0 (2.96 MB)
[345.11s] Trained multirockethydra-bestk-p-ridgecv_r0 in 10.3810s for f-0/r-0 (2.96 MB)
[345.15s] Trained multirockethydra-bestk-p-ridgecv_r0 in 10.4492s for f-2/r-0 (2.96 MB)
[345.76s] Trained multirockethydra-bestk-p-ridgecv_r0 in 11.0003s 

,random_state,492
,n_repetitions,1
,k_folds,10
,n_jobs,16
,keep_features,True
,verbose,2


In [5]:
preds = m.predict_per_model(X_test)
tests_accs = []
for model_name, model_preds in preds.items():
    acc = accuracy_score(y_test, model_preds)
    tests_accs.append({"model": model_name, "test_accuracy": acc})
test_df = pl.DataFrame(tests_accs)

[0.0000s] Starting prediction
Starting executor with 16 workers, run_dir=./tscglue/8be2aac49ff44aba
[105.0633s] Computed features for prediction
[105.3107s] Feature arrays saved to mmap files
[105.3111s] Starting prediction with 16 workers for 40 first-level models
[117.2310s] Predicted quant-etc_r0 in 0.0817s
[117.2793s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.5744s
[117.3162s] Predicted quant-etc_r0 in 0.0628s
[117.4154s] Predicted quant-etc_r0 in 0.0761s
[117.4278s] Predicted quant-etc_r0 in 0.0701s
[117.5080s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.5634s
[117.5248s] Predicted quant-etc_r0 in 0.0627s
[117.7330s] Predicted rdst-p-ridgecv_r0 in 0.1971s
[117.7650s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.6891s
[117.7769s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.7388s
[117.8614s] Predicted rdst-p-ridgecv_r0 in 0.3483s
[117.9094s] Predicted rdst-p-ridgecv_r0 in 0.4137s
[117.9820s] Predicted rdst-p-ridgecv_r0 in 0.1860s
[118.0860s] Predicted rd

In [6]:
pl.DataFrame(m.summary()).join(test_df, on="model").sort("test_accuracy", descending=True)

model,level,oof_accuracy,test_accuracy
str,i64,f64,f64
"""rdst-p-ridgecv_r0""",0,0.951,0.956757
"""probability-et""",1,0.953,0.956757
"""probability-rf""",1,0.958,0.951351
"""probability-ridgecv""",1,0.954,0.945946
"""multirockethydra-bestk-p-ridge…",0,0.954,0.940541
"""quant-etc_r0""",0,0.922,0.924324
"""rstsf_r0""",0,0.901,0.916216


In [7]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.0000s] Starting prediction
Starting executor with 16 workers, run_dir=./tscglue/8be2aac49ff44aba
[99.9768s] Computed features for prediction
[100.1618s] Feature arrays saved to mmap files
[100.1620s] Starting prediction with 16 workers for 40 first-level models
[111.6759s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.5099s
[111.7409s] Predicted quant-etc_r0 in 0.0523s
[111.7725s] Predicted quant-etc_r0 in 0.0448s
[111.8036s] Predicted quant-etc_r0 in 0.0507s
[111.8295s] Predicted quant-etc_r0 in 0.0444s
[111.8673s] Predicted quant-etc_r0 in 0.0491s
[111.9380s] Predicted quant-etc_r0 in 0.0738s
[112.0844s] Predicted rdst-p-ridgecv_r0 in 0.1842s
[112.0944s] Predicted rdst-p-ridgecv_r0 in 0.2071s
[112.3044s] Predicted rdst-p-ridgecv_r0 in 0.3037s
[112.3278s] Predicted rdst-p-ridgecv_r0 in 0.1846s
[112.4274s] Predicted rdst-p-ridgecv_r0 in 0.3141s
[112.5407s] Predicted rdst-p-ridgecv_r0 in 0.2206s
[112.7214s] Predicted multirockethydra-bestk-p-ridgecv_r0 in 0.6816s
[112.7363s] Pre

In [8]:
F = m.get_features()
F.estimated_size('gb')

0.8416101336479187

In [9]:
P = m.get_oof_predictions()
P.estimated_size('gb')

0.0001043081283569336

In [10]:
m = LokyStackerV6(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train, y_train)

[0.0000s] Starting fitting
Starting executor with 16 workers, run_dir=./tscglue/64b17c291585456c
[4.1594s] Computed QUANT features (1000, 24015) (183.22 MB) in 4.1582s
[4.1795s] Starting repetition 0
[13.6748s] Computed MultiRocket features (1000, 49728) (379.39 MB) in 9.4952s
[54.0459s] Computed Hydra features (1000, 9216) (70.31 MB) in 40.3708s
[258.9999s] Computed RDST features (1000, 30000) (228.88 MB) in 204.9535s
[259.0084s] Starting training with 16 workers for 40 models
[275.8836s] Trained multirockethydra-ridgecv in 7.1137s for f-6/r-0 (1.80 MB)
[276.1192s] Trained multirockethydra-ridgecv in 6.9243s for f-1/r-0 (1.80 MB)
[277.0108s] Trained multirockethydra-ridgecv in 7.3708s for f-9/r-0 (1.80 MB)
[277.6569s] Trained multirockethydra-ridgecv in 7.1325s for f-2/r-0 (1.80 MB)
[277.6658s] Trained multirockethydra-ridgecv in 7.7848s for f-8/r-0 (1.80 MB)
[277.8525s] Trained multirockethydra-ridgecv in 8.3312s for f-5/r-0 (1.80 MB)
[278.1340s] Trained multirockethydra-ridgecv in 7

,random_state,492
,n_repetitions,1
,k_folds,10
,time_limit_in_seconds,None
,n_jobs,16
,keep_features,False


In [11]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.0000s] Starting prediction
Starting executor with 16 workers, run_dir=./tscglue/64b17c291585456c
[99.1464s] Computed features for prediction
[99.3215s] Feature arrays saved to mmap files
[99.3218s] Starting prediction with 16 workers for 40 first-level models
[108.9781s] Predicted quant-etc in 0.0646s
[109.0649s] Predicted quant-etc in 0.0655s
[109.1529s] Predicted quant-etc in 0.0645s
[109.1987s] Predicted quant-etc in 0.0634s
[109.2449s] Predicted quant-etc in 0.0701s
[109.2919s] Predicted quant-etc in 0.0743s
[109.5085s] Predicted rdst-ridgecv in 0.2244s
[109.5783s] Predicted rdst-ridgecv in 0.2648s
[109.7179s] Predicted quant-etc in 0.0626s
[109.7819s] Predicted rdst-ridgecv in 0.2321s
[109.8513s] Predicted rdst-ridgecv in 0.2255s
[109.9757s] Predicted quant-etc in 0.0639s
[110.0190s] Predicted quant-etc in 0.0735s
[110.0504s] Predicted rdst-ridgecv in 0.3131s
[110.0663s] Predicted rdst-ridgecv in 0.2413s
[110.1767s] Predicted rdst-ridgecv in 0.3039s
[110.2056s] Predicted rdst-r

In [12]:
m = LokyStackerV5(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train, y_train)

[0.0000s] Starting fitting
[11.3224s] Computed QUANT features in 11.3216s
Starting executor with 16 workers, tmpdir=/tmp/loky_stacker_9rv0u62y
[11.3235s] Starting repetition 0
[20.6204s] Computed MultiRocket features in 9.2969s
[55.9904s] Computed Hydra features in 35.3696s
[273.6868s] Computed RDST features in 217.6962s
[274.4979s] Data saved to mmap files
[274.4982s] Starting training with 16 workers for 40 models
[300.3403s] Trained multirockethydra-ridgecv in 14.8406s for f-3/r-0
[300.5107s] Trained multirockethydra-ridgecv in 14.6707s for f-6/r-0
[300.7492s] Trained multirockethydra-ridgecv in 15.2002s for f-2/r-0
[301.4344s] Trained multirockethydra-ridgecv in 15.6050s for f-7/r-0
[301.7034s] Trained multirockethydra-ridgecv in 16.3671s for f-1/r-0
[301.8145s] Trained multirockethydra-ridgecv in 15.8863s for f-4/r-0
[301.9123s] Trained multirockethydra-ridgecv in 16.1098s for f-5/r-0
[301.9993s] Trained multirockethydra-ridgecv in 16.7901s for f-0/r-0
[302.2455s] Trained multiroc

,random_state,492
,n_repetitions,1
,k_folds,10
,time_limit_in_seconds,None
,n_jobs,16


In [13]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.0000s] Starting prediction
Starting executor with 16 workers, tmpdir=/tmp/loky_stacker_pred_4nguju_w
[104.7924s] Computed features for prediction
[105.1415s] Data saved to mmap files
[106.0537s] Starting prediction with 16 workers for 40 first-level models
[118.5226s] Predicted quant-etc in 0.5398s
[119.0212s] Predicted quant-etc in 0.4468s
[119.0419s] Predicted quant-etc in 0.4901s
[119.0627s] Predicted quant-etc in 0.4783s
[119.1014s] Predicted quant-etc in 0.6069s
[119.3775s] Predicted quant-etc in 0.5549s
[119.4510s] Predicted quant-etc in 0.5123s
[119.4864s] Predicted quant-etc in 0.4074s
[119.5434s] Predicted quant-etc in 0.4876s
[119.5793s] Predicted quant-etc in 0.4819s
[119.9422s] Predicted rdst-ridgecv in 0.7916s
[120.2966s] Predicted rdst-ridgecv in 0.7704s
[120.3407s] Predicted rdst-ridgecv in 0.8902s
[120.3964s] Predicted rdst-ridgecv in 0.8864s
[120.4784s] Predicted rdst-ridgecv in 0.8553s
[120.5843s] Predicted rdst-ridgecv in 0.9982s
[120.8933s] Predicted rdst-ridgecv